# DAGs Basics - Solutions

Complete solutions to the exercises in `dags_exercises.ipynb`.

---

## Setup

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from typing import List, Tuple
from sklearn.linear_model import LinearRegression

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
np.random.seed(42)

In [ ]:
# Helper function from the main notebook
def draw_dag(edges: List[Tuple[str, str]], title: str = "DAG", node_size: int = 3000, 
             font_size: int = 12, figsize: Tuple[int, int] = (8, 6)):
    """
    Draw a directed acyclic graph.
    """
    G = nx.DiGraph()
    G.add_edges_from(edges)
    
    plt.figure(figsize=figsize)
    pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
    
    nx.draw(G, pos, 
            with_labels=True,
            node_color='lightblue',
            node_size=node_size,
            font_size=font_size,
            font_weight='bold',
            arrows=True,
            arrowsize=20,
            arrowstyle='->',
            edge_color='gray',
            width=2,
            connectionstyle='arc3,rad=0.1')
    
    plt.title(title, fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()
    
    return G

---

## Exercise 1 Solution: Draw a DAG for Your Domain

**Example**: Does online advertising increase sales?

In [ ]:
# Define edges for advertising → sales DAG
edges_advertising = [
    # Confounders
    ('Company_Size', 'Advertising'),
    ('Company_Size', 'Sales'),
    ('Product_Quality', 'Advertising'),
    ('Product_Quality', 'Sales'),
    
    # Direct effect
    ('Advertising', 'Sales'),
    
    # Mediator
    ('Advertising', 'Website_Traffic'),
    ('Website_Traffic', 'Sales'),
    
    # Collider
    ('Advertising', 'Profit'),
    ('Sales', 'Profit')
]

G_advertising = draw_dag(edges_advertising, 
                         title="Effect of Advertising on Sales",
                         figsize=(10, 7))

### Analysis

**1. Confounders** (must control):
- **Company_Size**: Larger companies both advertise more AND have higher sales
- **Product_Quality**: Better products lead to more advertising budget AND higher sales

**2. Mediators** (must NOT control for total effect):
- **Website_Traffic**: Advertising works through increasing traffic, which increases sales
- Controlling for traffic would block the mechanism

**3. Colliders** (must NOT control):
- **Profit**: Affected by both advertising (costs) and sales (revenue)
- Conditioning on profit creates selection bias

**4. Correct adjustment set**:
- Control for: {Company_Size, Product_Quality}
- Do NOT control for: Website_Traffic, Profit

**Backdoor paths to block**:
1. Advertising ← Company_Size → Sales
2. Advertising ← Product_Quality → Sales

Both blocked by controlling for Company_Size and Product_Quality.

---

## Exercise 2 Solution: Simpson's Paradox

Create a simulation where the naive and adjusted effects have opposite signs.

In [ ]:
def simulate_simpsons_paradox(n: int = 1000) -> pd.DataFrame:
    """
    Simulate data where naive and adjusted effects have opposite signs.
    
    DAG: Treatment ← Confounder → Outcome, Treatment → Outcome
    
    Structure:
    - Confounder has STRONG positive effect on Treatment
    - Confounder has STRONG negative effect on Outcome
    - Treatment has small positive effect on Outcome
    
    Result: Naive estimate is negative, true effect is positive!
    """
    np.random.seed(42)
    
    # Confounder (e.g., stress level)
    confounder = np.random.normal(0, 1, n)
    
    # Treatment (e.g., coffee consumption)
    # High stress → More coffee
    treatment = 5 + 3 * confounder + np.random.normal(0, 0.5, n)
    
    # Outcome (e.g., health score)
    # High stress → Lower health (strong negative effect)
    # Coffee → Slightly better health (small positive effect)
    outcome = 100 - 10 * confounder + 0.5 * treatment + np.random.normal(0, 2, n)
    
    return pd.DataFrame({
        'confounder': confounder,
        'treatment': treatment,
        'outcome': outcome
    })

# Generate data
df = simulate_simpsons_paradox()

print("Structural equations:")
print("  Confounder (Stress) ~ N(0, 1)")
print("  Treatment (Coffee) = 5 + 3*Stress + ε₁")
print("  Outcome (Health) = 100 - 10*Stress + 0.5*Coffee + ε₂")
print("\nTrue causal effect of Coffee on Health: +0.5")
print("\nFirst 5 rows:")
print(df.head())

In [ ]:
# Calculate naive estimate (biased)
X_naive = df[['treatment']].values
y = df['outcome'].values

model_naive = LinearRegression()
model_naive.fit(X_naive, y)
coef_naive = model_naive.coef_[0]

# Calculate adjusted estimate (correct)
X_adjusted = df[['treatment', 'confounder']].values

model_adjusted = LinearRegression()
model_adjusted.fit(X_adjusted, y)
coef_adjusted = model_adjusted.coef_[0]  # Coefficient for treatment

print("\n" + "="*60)
print("SIMPSON'S PARADOX DEMONSTRATION")
print("="*60)
print(f"\nTRUE causal effect: +0.5")
print(f"\nNaive estimate (NO control):     {coef_naive:+.3f}  ← NEGATIVE!")
print(f"Adjusted estimate (control):     {coef_adjusted:+.3f}  ← POSITIVE!")
print(f"\nThe sign REVERSED! This is Simpson's Paradox.")
print(f"\nBias from omitting confounder: {coef_naive - coef_adjusted:.3f}")

print("\n🔑 Explanation:")
print("  • Stressed people drink MORE coffee (positive correlation)")
print("  • Stressed people have WORSE health (negative correlation)")
print("  • Without controlling for stress, coffee appears harmful")
print("  • After controlling for stress, coffee's true (small) benefit emerges")

In [ ]:
# Visualize Simpson's Paradox
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Naive relationship (appears negative)
axes[0].scatter(df['treatment'], df['outcome'], alpha=0.5, s=30, color='gray')
x_range = np.array([df['treatment'].min(), df['treatment'].max()])
axes[0].plot(x_range, model_naive.predict(x_range.reshape(-1, 1)), 
             'r-', linewidth=3, label=f'Naive slope: {coef_naive:.2f} (NEGATIVE!)')
axes[0].set_xlabel('Treatment (Coffee consumption)', fontsize=11)
axes[0].set_ylabel('Outcome (Health score)', fontsize=11)
axes[0].set_title('Naive Relationship (Confounded)\nCoffee appears HARMFUL', fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Right: Colored by confounder (shows reversal)
scatter = axes[1].scatter(df['treatment'], df['outcome'], 
                          c=df['confounder'], cmap='RdYlGn_r',  # Red = high stress
                          alpha=0.7, s=40, edgecolors='black', linewidth=0.5)

# Add true causal effect line
# At mean stress level, the relationship is positive
mean_stress = df['confounder'].mean()
intercept_adjusted = 100 - 10 * mean_stress + model_adjusted.coef_[1] * mean_stress
axes[1].plot(x_range, intercept_adjusted + coef_adjusted * x_range,
             'b--', linewidth=3, label=f'True slope: {coef_adjusted:.2f} (POSITIVE!)')

axes[1].set_xlabel('Treatment (Coffee consumption)', fontsize=11)
axes[1].set_ylabel('Outcome (Health score)', fontsize=11)
axes[1].set_title('Colored by Confounder (Stress)\nCoffee is actually BENEFICIAL', fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

cbar = plt.colorbar(scatter, ax=axes[1])
cbar.set_label('Stress Level\n(Red = High)', rotation=270, labelpad=20, fontsize=10)

plt.tight_layout()
plt.show()

print("\nVisualization interpretation:")
print("  • Red points (high stress): Low health despite varying coffee intake")
print("  • Green points (low stress): High health across coffee levels")
print("  • Within each stress level, coffee has a small positive effect")
print("  • But across all data, the negative naive slope emerges due to confounding")

### Discussion

**Simpson's Paradox** occurs when:
1. A strong confounder affects both treatment and outcome
2. The confounder's effects on treatment and outcome have opposite signs (or very different magnitudes)
3. This creates a spurious association that can even reverse the true causal direction

**Real-world examples**:
- **UC Berkeley admissions bias case**: Overall admission rates appeared biased against women, but within each department, there was no bias (women applied to more competitive departments)
- **Hospital death rates**: Better hospitals may have higher death rates because they treat sicker patients
- **Drug effectiveness**: A treatment may appear ineffective overall but work well in each subgroup

**The lesson**: Always consider potential confounders before interpreting correlations!

---

## Exercise 3 Solution: M-Bias Structure

Demonstrate how controlling for a non-confounder can introduce bias.

In [ ]:
# Draw the M-bias DAG
edges_mbias = [
    ('U1', 'X'),
    ('U1', 'Z'),
    ('U2', 'Y'),
    ('U2', 'Z')
]

G_mbias = draw_dag(edges_mbias, title="M-Bias Structure", figsize=(10, 6))

print("\nM-Bias Structure:")
print("  U1 → X     U2 → Y")
print("   ↓           ↓")
print("   → Z ←")
print("\nKey features:")
print("  • X and Y are INDEPENDENT (no path between them)")
print("  • Z is a COLLIDER (affected by both U1 and U2)")
print("  • U1 and U2 are unmeasured")
print("  • Controlling for Z opens the path: X ← U1 → Z ← U2 → Y")

In [ ]:
def simulate_mbias(n: int = 1000) -> pd.DataFrame:
    """
    Simulate M-bias structure where X and Y are independent,
    but become associated if we condition on Z.
    
    Example interpretation:
    - X = Drug A usage (affected by patient preference U1)
    - Y = Recovery (affected by disease severity U2)
    - Z = Hospital admission (affected by both patient preference and disease severity)
    - U1, U2 = unmeasured variables
    """
    np.random.seed(42)
    
    # Unmeasured variables (independent)
    u1 = np.random.normal(0, 1, n)  # e.g., patient preference
    u2 = np.random.normal(0, 1, n)  # e.g., disease severity
    
    # X depends only on U1 (no effect of Y)
    x = 2 * u1 + np.random.normal(0, 0.5, n)
    
    # Y depends only on U2 (no effect of X)
    y = 3 * u2 + np.random.normal(0, 0.5, n)
    
    # Z depends on both U1 and U2 (Z is a COLLIDER)
    z = u1 + u2 + np.random.normal(0, 0.5, n)
    
    return pd.DataFrame({
        'u1': u1,
        'u2': u2,
        'x': x,
        'y': y,
        'z': z
    })

df_mbias = simulate_mbias()

print("M-Bias Simulation:")
print("\nStructural equations:")
print("  U1, U2 ~ N(0, 1)  [independent]")
print("  X = 2*U1 + ε₁     [no Y in equation!]")
print("  Y = 3*U2 + ε₂     [no X in equation!]")
print("  Z = U1 + U2 + ε₃  [collider]")
print("\nTrue causal effect of X on Y: 0 (they're independent!)")
print("\nFirst 5 rows:")
print(df_mbias.head())

In [ ]:
# Estimate effects

# Model 1: Y ~ X (correct - no confounding)
X_no_z = df_mbias[['x']].values
y_mbias = df_mbias['y'].values

model_no_z = LinearRegression()
model_no_z.fit(X_no_z, y_mbias)
coef_no_z = model_no_z.coef_[0]

# Model 2: Y ~ X + Z (biased - conditioning on collider!)
X_with_z = df_mbias[['x', 'z']].values

model_with_z = LinearRegression()
model_with_z.fit(X_with_z, y_mbias)
coef_with_z = model_with_z.coef_[0]  # Coefficient for X

# Check correlation
corr_x_y = df_mbias['x'].corr(df_mbias['y'])

print("\n" + "="*60)
print("M-BIAS DEMONSTRATION")
print("="*60)
print(f"\nTRUE causal effect of X on Y: 0.0 (they're independent!)")
print(f"\nCorrelation(X, Y): {corr_x_y:.3f} (near zero ✓)")
print(f"\nEffect estimate WITHOUT controlling for Z: {coef_no_z:+.3f}  ← CORRECT (near 0)")
print(f"Effect estimate WITH controlling for Z:    {coef_with_z:+.3f}  ← BIASED!")

print("\n🔑 Key Insight:")
print("  • X and Y are truly independent (no causal relationship)")
print("  • Z is a collider: both U1 and U2 cause Z")
print("  • Controlling for Z creates a spurious association")
print("  • This is M-bias: controlling introduces bias where none existed!")

print("\n⚠️  Lesson:")
print("  • Don't control for variables 'just to be safe'")
print("  • Controlling for non-confounders can INTRODUCE bias")
print("  • Use a DAG to decide what to control for")

In [ ]:
# Visualize M-bias
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Left: X vs Y (no association)
axes[0].scatter(df_mbias['x'], df_mbias['y'], alpha=0.5, s=30)
axes[0].axhline(y=df_mbias['y'].mean(), color='r', linestyle='--', linewidth=2, 
                label=f'Slope ≈ {coef_no_z:.2f}')
axes[0].set_xlabel('X', fontsize=11)
axes[0].set_ylabel('Y', fontsize=11)
axes[0].set_title('X and Y: No Association\n(Correct!)', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Middle: X vs Y colored by Z (shows how Z is affected by both U1 and U2)
scatter = axes[1].scatter(df_mbias['x'], df_mbias['y'], 
                          c=df_mbias['z'], cmap='viridis',
                          alpha=0.7, s=40, edgecolors='black', linewidth=0.5)
axes[1].set_xlabel('X', fontsize=11)
axes[1].set_ylabel('Y', fontsize=11)
axes[1].set_title('Colored by Z (the Collider)', fontweight='bold')
axes[1].grid(True, alpha=0.3)
cbar = plt.colorbar(scatter, ax=axes[1])
cbar.set_label('Z value', rotation=270, labelpad=15)

# Right: Show association within strata of Z
# Stratify by Z into low/medium/high
df_mbias['z_category'] = pd.qcut(df_mbias['z'], q=3, labels=['Low Z', 'Medium Z', 'High Z'])

colors = ['red', 'orange', 'green']
for i, (category, color) in enumerate(zip(['Low Z', 'Medium Z', 'High Z'], colors)):
    subset = df_mbias[df_mbias['z_category'] == category]
    axes[2].scatter(subset['x'], subset['y'], alpha=0.6, s=30, 
                   color=color, label=category)
    
    # Add trend line for each stratum
    z_slope = np.polyfit(subset['x'], subset['y'], 1)[0]
    z_poly = np.poly1d(np.polyfit(subset['x'], subset['y'], 1))
    x_range_z = np.linspace(subset['x'].min(), subset['x'].max(), 100)
    axes[2].plot(x_range_z, z_poly(x_range_z), color=color, linewidth=2, alpha=0.8)

axes[2].set_xlabel('X', fontsize=11)
axes[2].set_ylabel('Y', fontsize=11)
axes[2].set_title('Within Z Strata: Spurious Association\n(M-bias!)', fontweight='bold')
axes[2].legend(fontsize=9)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nVisualization interpretation:")
print("  • Left: X and Y show no relationship (correct)")
print("  • Middle: Z is influenced by both U1 (→X) and U2 (→Y)")
print("  • Right: Within each Z stratum, X and Y appear associated (spurious!)")
print("  • Conditioning on Z (the collider) creates bias")

### Explanation

**1. Why are X and Y independent in this structure?**
- There is no directed path from X to Y or Y to X
- U1 only affects X (not Y)
- U2 only affects Y (not X)
- U1 and U2 are independent
- Therefore, X and Y are independent

**2. Why does controlling for Z create bias?**
- Z is a **collider**: both U1 and U2 cause Z
- Conditioning on a collider opens a path between its causes
- This creates the backdoor path: X ← U1 → Z ← U2 → Y
- Even though U1 and U2 are independent, conditioning on their common effect Z makes them dependent
- This induced dependency between U1 and U2 creates a spurious association between X and Y

**3. What's the lesson for causal inference?**
- **Don't control for variables blindly!** "Control for everything" is bad advice
- Use a DAG to identify what variables are confounders vs colliders vs mediators
- Controlling for colliders introduces bias (M-bias is one example)
- You need domain knowledge to draw the correct DAG
- When in doubt, think carefully about causal structure before adding controls

---

## Exercise 4 Solution: Finding Multiple Paths

Create a complex DAG and identify all paths programmatically.

In [ ]:
# Define a complex DAG with multiple paths
edges_complex = [
    # Confounders
    ('Confounder1', 'X'),
    ('Confounder1', 'Y'),
    ('Confounder2', 'X'),
    ('Confounder2', 'Mediator'),
    
    # Direct effect
    ('X', 'Y'),
    
    # Mediated paths
    ('X', 'Mediator'),
    ('Mediator', 'Y'),
    
    # Additional complexity
    ('Confounder1', 'Confounder2')
]

G_complex = draw_dag(edges_complex, title="Complex DAG with Multiple Paths", figsize=(10, 7))

print("\nDAG Structure:")
print("  Confounder1 → X, Y, Confounder2")
print("  Confounder2 → X, Mediator")
print("  X → Y, Mediator")
print("  Mediator → Y")

In [ ]:
# Implement path-finding function
def find_all_paths(G: nx.DiGraph, source: str, target: str, undirected: bool = True) -> List[List[str]]:
    """
    Find all paths between source and target.
    
    Parameters
    ----------
    G : nx.DiGraph
        Directed graph
    source : str
        Source node
    target : str
        Target node
    undirected : bool
        If True, treat edges as undirected (finds all paths including backdoor)
        If False, only follow directed edges (finds only causal paths)
        
    Returns
    -------
    List[List[str]]
        List of paths, where each path is a list of nodes
    """
    if undirected:
        # Convert to undirected to find all paths (including backdoor)
        G_undirected = G.to_undirected()
        paths = list(nx.all_simple_paths(G_undirected, source, target))
    else:
        # Only directed paths (causal)
        try:
            paths = list(nx.all_simple_paths(G, source, target))
        except nx.NodeNotFound:
            paths = []
    
    return paths

# Find all paths and directed paths
all_paths = find_all_paths(G_complex, 'X', 'Y', undirected=True)
directed_paths = find_all_paths(G_complex, 'X', 'Y', undirected=False)

print("\n" + "="*60)
print("PATH ANALYSIS: X → Y")
print("="*60)

print("\nAll paths from X to Y (treating edges as undirected):")
for i, path in enumerate(all_paths, 1):
    path_str = ' - '.join(path)
    print(f"{i}. {path_str}")

print(f"\nTotal: {len(all_paths)} paths")

print("\n" + "-"*60)
print("\nDirected (causal) paths from X to Y:")
for i, path in enumerate(directed_paths, 1):
    path_str = ' → '.join(path)
    print(f"{i}. {path_str}")

print(f"\nTotal: {len(directed_paths)} causal paths")

In [ ]:
# Identify path types
def classify_path(G: nx.DiGraph, path: List[str]) -> str:
    """
    Classify a path as causal or backdoor.
    
    A path is causal if all edges follow the arrow direction.
    A path is backdoor if at least one edge goes against the arrow direction.
    """
    for i in range(len(path) - 1):
        # Check if edge exists in the directed graph
        if not G.has_edge(path[i], path[i+1]):
            # Edge goes backwards
            return "Backdoor"
    return "Causal"

print("\n" + "="*60)
print("PATH CLASSIFICATION")
print("="*60)

causal_paths = []
backdoor_paths = []

for path in all_paths:
    path_type = classify_path(G_complex, path)
    if path_type == "Causal":
        causal_paths.append(path)
    else:
        backdoor_paths.append(path)

print("\n✅ CAUSAL PATHS (follow arrow directions):")
for i, path in enumerate(causal_paths, 1):
    path_str = ' → '.join(path)
    print(f"{i}. {path_str}")

print("\n⚠️  BACKDOOR PATHS (confounding - go against arrows):")
for i, path in enumerate(backdoor_paths, 1):
    # Show direction more clearly
    path_parts = []
    for j in range(len(path) - 1):
        if G_complex.has_edge(path[j], path[j+1]):
            path_parts.append(f"{path[j]} →")
        else:
            path_parts.append(f"{path[j]} ←")
    path_parts.append(path[-1])
    path_str = ' '.join(path_parts)
    print(f"{i}. {path_str}")

### Path Analysis

**Causal paths** (follow arrow directions):
1. X → Y (direct effect)
2. X → Mediator → Y (indirect/mediated effect)

**Backdoor paths** (confounding - go against arrows):
1. X ← Confounder1 → Y
2. X ← Confounder2 → Mediator → Y
3. X ← Confounder1 → Confounder2 → Mediator → Y

**Minimal adjustment set** (variables to control for):
- **Confounder1** alone is sufficient!

**Explanation**: Why does controlling for Confounder1 block all backdoor paths?

Let's check each backdoor path:
1. X ← **Confounder1** → Y
   - Blocked by controlling for Confounder1 ✓

2. X ← Confounder2 → Mediator → Y
   - This path goes: X ← Confounder2, but Confounder2 ← Confounder1
   - Controlling for Confounder1 blocks the path before it reaches Confounder2 ✓

3. X ← **Confounder1** → Confounder2 → Mediator → Y
   - Blocked by controlling for Confounder1 ✓

**Alternative adjustment sets**:
- {Confounder1} - minimal
- {Confounder1, Confounder2} - also works but includes unnecessary variable
- {Confounder2} - NOT sufficient (doesn't block path 1)

**What NOT to control for**:
- **Mediator**: Would block the causal path X → Mediator → Y

In [ ]:
# BONUS: Simulate data and verify adjustment set
def simulate_complex_dag(n: int = 1000) -> pd.DataFrame:
    """
    Simulate data following the complex DAG.
    
    True causal effect of X on Y:
    - Direct: X → Y (coefficient = 2)
    - Indirect: X → Mediator → Y (coefficient = 1 * 1.5 = 1.5)
    - Total effect: 2 + 1.5 = 3.5
    """
    np.random.seed(42)
    
    # Confounder1 (root cause)
    conf1 = np.random.normal(0, 1, n)
    
    # Confounder2 (affected by Confounder1)
    conf2 = 0.8 * conf1 + np.random.normal(0, 0.5, n)
    
    # X (treatment) - affected by both confounders
    x = 3 * conf1 + 2 * conf2 + np.random.normal(0, 1, n)
    
    # Mediator - affected by X and Confounder2
    mediator = 1.0 * x + 1.5 * conf2 + np.random.normal(0, 0.5, n)
    
    # Y (outcome) - affected by X, Mediator, and Confounder1
    y = 2.0 * x + 1.5 * mediator + 4 * conf1 + np.random.normal(0, 1, n)
    
    return pd.DataFrame({
        'confounder1': conf1,
        'confounder2': conf2,
        'x': x,
        'mediator': mediator,
        'y': y
    })

df_complex = simulate_complex_dag()

print("\n" + "="*60)
print("SIMULATION VERIFICATION")
print("="*60)

print("\nTrue causal effects:")
print("  Direct effect (X → Y): 2.0")
print("  Indirect effect (X → Mediator → Y): 1.0 × 1.5 = 1.5")
print("  Total effect: 2.0 + 1.5 = 3.5")

# Model 1: Naive (no controls)
X1 = df_complex[['x']].values
y_complex = df_complex['y'].values
model1 = LinearRegression().fit(X1, y_complex)
coef1 = model1.coef_[0]

# Model 2: Control for Confounder1 only (correct!)
X2 = df_complex[['x', 'confounder1']].values
model2 = LinearRegression().fit(X2, y_complex)
coef2 = model2.coef_[0]

# Model 3: Control for both confounders (also works, but includes extra variable)
X3 = df_complex[['x', 'confounder1', 'confounder2']].values
model3 = LinearRegression().fit(X3, y_complex)
coef3 = model3.coef_[0]

# Model 4: Wrong - control for Mediator (blocks causal path)
X4 = df_complex[['x', 'confounder1', 'mediator']].values
model4 = LinearRegression().fit(X4, y_complex)
coef4 = model4.coef_[0]

print("\n" + "-"*60)
print("\nEstimated effects:")
print(f"\n1. Naive (no controls):              {coef1:.2f}  ← BIASED (confounded)")
print(f"2. Control for Confounder1:          {coef2:.2f}  ← CORRECT! ✓")
print(f"3. Control for both confounders:     {coef3:.2f}  ← Also correct (redundant)")
print(f"4. Control for Confounder1+Mediator: {coef4:.2f}  ← WRONG (blocks mediation)")

print("\n🔑 Key Insights:")
print("  • Naive estimate is biased upward (confounding adds positive bias)")
print("  • Controlling for Confounder1 alone recovers the true total effect (3.5)")
print("  • Controlling for both confounders also works (but Confounder2 is redundant)")
print("  • Controlling for Mediator blocks the indirect path, giving only direct effect (2.0)")
print("\n  ✅ Minimal adjustment set {Confounder1} is VERIFIED!")

---

## Summary

These exercises demonstrate key DAG concepts:

**Exercise 1**: Drawing DAGs forces you to think through causal structure and identify confounders, mediators, and colliders.

**Exercise 2**: Simpson's Paradox shows how strong confounding can reverse the apparent direction of an effect. Always consider confounders!

**Exercise 3**: M-bias demonstrates that controlling for variables can introduce bias. Don't control for colliders or their descendants.

**Exercise 4**: Complex DAGs require systematic path analysis to identify the minimal adjustment set. Use the graph structure to determine what to control for.

### General Principles

1. **Draw the DAG first** - Make your causal assumptions explicit
2. **Identify path types** - Distinguish causal paths from backdoor paths
3. **Find adjustment sets** - Control for variables that block backdoor paths
4. **Don't over-control** - Avoid controlling for mediators and colliders
5. **Verify with simulation** - When possible, test your reasoning with simulated data

### Moving Forward

- Practice drawing DAGs for research questions in your domain
- Use tools like DAGitty (http://www.dagitty.net/) for complex DAGs
- Combine DAG reasoning with potential outcomes framework
- Always think about the causal structure before running regressions!